# BERT-style Transformer for South Park Character Classification

This notebook implements a custom BERT-style transformer encoder for predicting which South Park character spoke a given line of dialogue.

## Why Transformers?

Transformers have revolutionized NLP by addressing key limitations of RNNs:

- **Parallel Processing** - Unlike RNNs, transformers process entire sequences simultaneously
- **Long-range Dependencies** - Self-attention captures relationships between any two tokens
- **No Vanishing Gradients** - Direct connections between all positions
- **Bidirectional Context** - Understands context from both directions simultaneously

### BERT Architecture:

Our implementation follows BERT's encoder-only design:
1. **Token + Positional Embeddings** - Encode words and their positions
2. **Multi-Head Self-Attention** - Learn different aspects of relationships
3. **Feed-Forward Networks** - Process attended representations
4. **Layer Normalization + Residual Connections** - Stable training
5. **[CLS] Token Classification** - Use first token for sequence classification

For character classification, we expect transformers to capture:
- **Character-specific phrases** - Cartman: Respect my authority
- **Speaking patterns** - Stan: Dude, this is...
- **Contextual nuances** - Word order and emphasis matter

## 1. Imports and Setup

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.optim.lr_scheduler import OneCycleLR

import pandas as pd
import numpy as np

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import time
import math

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cpu


## 2. Configuration Parameters

In [4]:
# Transformer hyperparameters
EMBEDDING_DIM = 128
NUM_HEADS = 4
NUM_LAYERS = 4
HIDDEN_DIM = 512
MAX_SEQ_LEN = 256
DROPOUT_RATE = 0.2

# Training hyperparameters
BATCH_SIZE = 64
LEARNING_RATE = 1e-4
MAX_LR = 5e-4
NUM_EPOCHS = 50
WEIGHT_DECAY = 0.01
LABEL_SMOOTHING = 0.1

# Early stopping
EARLY_STOP_PATIENCE = 7

# Text preprocessing
MIN_WORD_FREQ = 2
MAX_VOCAB_SIZE = 10000



print(f"Configuration:")
print(f"  Embedding dim: {EMBEDDING_DIM}")
print(f"  Num attention heads: {NUM_HEADS}")
print(f"  Num transformer layers: {NUM_LAYERS}")
print(f"  Feedforward dim: {HIDDEN_DIM}")
print(f"  Max sequence length: {MAX_SEQ_LEN}")
print(f"  Dropout rate: {DROPOUT_RATE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE} -> {MAX_LR}")
print(f"  Max epochs: {NUM_EPOCHS}")
print(f"  Early stopping patience: {EARLY_STOP_PATIENCE}")
print(f"  Label smoothing: {LABEL_SMOOTHING}")

Configuration:
  Embedding dim: 128
  Num attention heads: 4
  Num transformer layers: 4
  Feedforward dim: 512
  Max sequence length: 256
  Dropout rate: 0.2
  Batch size: 64
  Learning rate: 0.0001 -> 0.0005
  Max epochs: 50
  Early stopping patience: 7
  Label smoothing: 0.1


## 3. Load Data

In [5]:
import pandas as pd
# read data from parquet 
df = pd.read_parquet('../01 - Inputs/south_park_dialogues.parquet')

In [ ]:
train_data = pd.read_parquet('../01 - Inputs/train_data_labels_as_numbers.parquet')
val_data = pd.read_parquet('../01 - Inputs/val_data_labels_as_numbers.parquet')
test_data = pd.read_parquet('../01 - Inputs/test_data_labels_as_numbers.parquet')

## 6. Text Preprocessing and Vocabulary

For BERT-style transformers, we need special tokens:
- `<pad>` - Padding token (index 0)
- `<unk>` - Unknown words (index 1)
- `<cls>` - Classification token (index 2) - prepended to every sequence
- `<sep>` - Separator token (index 3) - optional for single sentences

In [10]:
train_texts = train_data['text'].tolist()
train_labels = train_data['label'].tolist()

val_texts = val_data['text'].tolist()
val_labels = val_data['label'].tolist()

test_texts = test_data['text'].tolist()
test_labels = test_data['label'].tolist()

In [ ]:
from model_utils import preprocess_text, Vocabulary


# Preprocess all texts
print("Preprocessing texts...")
train_tokens = [preprocess_text(text) for text in tqdm(train_texts, desc="Train")]
val_tokens = [preprocess_text(text) for text in tqdm(val_texts, desc="Val")]
test_tokens = [preprocess_text(text) for text in tqdm(test_texts, desc="Test")]

# Build vocabulary
vocab = Vocabulary(min_freq=MIN_WORD_FREQ, max_size=MAX_VOCAB_SIZE)
vocab.build_vocab(train_tokens)

print(f"\nVocabulary size: {len(vocab):,}")
print(f"Special tokens: <pad>={vocab.word2idx['<pad>']}, <unk>={vocab.word2idx['<unk>']}, <cls>={vocab.word2idx['<cls>']}, <sep>={vocab.word2idx['<sep>']}")
print(f"\nMost common words:")
for word, count in vocab.word_counts.most_common(10):
    print(f"  {word}: {count}")

Preprocessing texts...


Test: 100%|██████████| 12072/12072 [00:00<00:00, 207537.87it/s]



Vocabulary size: 5,364
Special tokens: <pad>=0, <unk>=1, <cls>=2, <sep>=3

Most common words:
  you: 5273
  the: 4561
  i: 4171
  to: 4046
  a: 2977
  and: 2602
  it: 2210
  we: 1940
  that: 1874
  is: 1840


## 7. Dataset and DataLoader\n
\n
For BERT-style models, we prepend the `<cls>` token to each sequence and create attention masks.

31

<function model_utils.create_dataloaders(train_tokens, train_labels, val_tokens, val_labels, test_tokens, test_labels, vocab, model_type, max_seq_len=256, batch_size=64, configs={'bert': {'dataset': <class 'datasets.BERTDialogueDataset'>, 'collate': <function collate_fn_bert at 0x00000195757ABA00>, 'needs_vocab': True, 'needs_max_len': True}, 'distilbert': {'dataset': <class 'datasets.BERTDialogueDataset'>, 'collate': <function collate_fn_bert at 0x00000195757ABA00>, 'needs_vocab': True, 'needs_max_len': True}, 'rnn': {'dataset': <class 'datasets.DialogueDataset'>, 'collate': <function collate_fn_rnn at 0x00000195757AB060>, 'needs_vocab': False, 'needs_max_len': False}, 'lstm': {'dataset': <class 'datasets.DialogueDataset'>, 'collate': <function collate_fn_rnn at 0x00000195757AB060>, 'needs_vocab': False, 'needs_max_len': False}, 'gru': {'dataset': <class 'datasets.DialogueDataset'>, 'collate': <function collate_fn_rnn at 0x00000195757AB060>, 'needs_vocab': False, 'needs_max_len': Fals

In [12]:
from model_utils import create_dataloaders

train_loader, val_loader, test_loader = create_dataloaders(train_tokens, train_labels, val_tokens, val_labels,
                       test_tokens, test_labels, vocab, 'bert')

print(f"Train batches: {len(train_loader)}  Val batches: {len(val_loader)}  Test batches: {len(test_loader)}")

# Test a batch
sample_batch = next(iter(train_loader))
print(f"\nSample batch shapes:")
print(f"  input_ids: {sample_batch['input_ids'].shape}")
print(f"  attention_mask: {sample_batch['attention_mask'].shape}")
print(f"  labels: {sample_batch['labels'].shape}")
print(f"\nFirst sequence (with [CLS]):")
print(f"  Tokens: {vocab.decode(sample_batch['input_ids'][0].tolist()[:20])}")

Train batches: 221  Val batches: 221  Test batches: 189

Sample batch shapes:
  input_ids: torch.Size([64, 51])
  attention_mask: torch.Size([64, 51])
  labels: torch.Size([64])

First sequence (with [CLS]):
  Tokens: ['<cls>', 'where', 'is', 'it', 'where', 'is', 'it', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>']


## 8. BERT-style Transformer Model Architecture\n
\n
We implement a custom BERT-style transformer encoder using PyTorch's built-in components:\n
\n
1. **Token Embeddings** - Convert word indices to dense vectors\n
2. **Positional Embeddings** - Add position information (learned, not sinusoidal)\n
3. **Transformer Encoder** - Multi-head self-attention + feedforward layers\n
4. **Classification Head** - Use [CLS] token representation for prediction\n
\n
Key differences from RNN/LSTM:\n
- **Parallel processing** - All tokens processed simultaneously\n
- **Self-attention** - Each token attends to all other tokens\n
- **Positional encoding** - Explicit position information (RNNs get this implicitly)\n
- **No recurrence** - No hidden state passed between time steps

In [18]:
class TransformerClassifier(nn.Module):
    """BERT-style transformer encoder for sequence classification."""
    
    def __init__(self, vocab_size, embedding_dim=128, num_heads=4, 
                 num_layers=4, hidden_dim=512, num_classes=12, 
                 max_seq_len=256, dropout=0.2):
        super(TransformerClassifier, self).__init__()
        
        self.embedding_dim = embedding_dim
        self.max_seq_len = max_seq_len
        
        # Token embeddings
        self.token_embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # Positional embeddings (learned, not sinusoidal like original Transformer)
        self.pos_embedding = nn.Embedding(max_seq_len, embedding_dim)
        
        # Dropout for embeddings
        self.embedding_dropout = nn.Dropout(dropout)
        
        # Transformer encoder layers
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim,
            dropout=dropout,
            activation='gelu',  # GELU activation like BERT
            batch_first=True,
            norm_first=False  # Post-norm like original Transformer
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Classification head (like BERT)
        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights following BERT initialization."""
        for module in self.modules():
            if isinstance(module, nn.Linear):
                module.weight.data.normal_(mean=0.0, std=0.02)
                if module.bias is not None:
                    module.bias.data.zero_()
            elif isinstance(module, nn.Embedding):
                module.weight.data.normal_(mean=0.0, std=0.02)
                if module.padding_idx is not None:
                    module.weight.data[module.padding_idx].zero_()
    
    def forward(self, input_ids, attention_mask=None):
        """
        Forward pass.
        
        Args:
            input_ids: (batch_size, seq_len) - Token indices
            attention_mask: (batch_size, seq_len) - 1 for real tokens, 0 for padding
        
        Returns:
            logits: (batch_size, num_classes) - Classification logits
        """
        batch_size, seq_len = input_ids.size()
        
        # Create position indices
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(batch_size, -1)
        
        # Token + positional embeddings
        token_embeds = self.token_embedding(input_ids)
        pos_embeds = self.pos_embedding(positions)
        embeddings = token_embeds + pos_embeds
        embeddings = self.embedding_dropout(embeddings)
        
        # Create padding mask for transformer (True for padding, False for real tokens)
        # PyTorch transformer expects True for positions to IGNORE
        if attention_mask is not None:
            padding_mask = (attention_mask == 0)
        else:
            padding_mask = None
        
        # Transformer encoding
        encoded = self.transformer(embeddings, src_key_padding_mask=padding_mask)
        
        # Use [CLS] token (first token) for classification
        cls_output = encoded[:, 0, :]  # (batch_size, embedding_dim)
        
        # Classification
        logits = self.classifier(cls_output)
        
        return logits


# Initialize model
model = TransformerClassifier(
    vocab_size=len(vocab),
    embedding_dim=EMBEDDING_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    hidden_dim=HIDDEN_DIM,
    num_classes=len(set(train_labels)),
    max_seq_len=MAX_SEQ_LEN,
    dropout=DROPOUT_RATE
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model initialized")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nModel architecture:")
print(model)

Model initialized
Total parameters: 1,594,399
Trainable parameters: 1,594,399

Model architecture:
TransformerClassifier(
  (token_embedding): Embedding(5364, 128, padding_idx=0)
  (pos_embedding): Embedding(256, 128)
  (embedding_dropout): Dropout(p=0.2, inplace=False)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.2, inplace=False)
        (dropout2): Dropout(p=0.2, inplace=False)
      )
    )
  )
  (classifier): Sequ

## 9. Class Weights

In [19]:
from train_utils import compute_balanced_class_weights

class_weights = compute_balanced_class_weights(train_labels, len(set(train_labels)), device)

print("Class weights computed")
print(f"Class weights: {class_weights}")
print(f"\nWeight range: {class_weights.min():.3f} - {class_weights.max():.3f}")

Class weights computed
Class weights: tensor([0.1364, 0.1860, 0.1982, 0.5031, 0.5031, 1.2179, 1.4944, 1.5143, 1.7273,
        1.9168, 2.0281, 2.0372, 2.2490, 2.3178, 2.3661, 2.3661, 3.1992, 3.4945,
        3.7545, 4.2064, 4.2064, 4.2858, 4.4979, 4.6356, 4.6834, 4.6834, 4.9922,
        5.4082, 5.9775, 6.1391, 6.2232])

Weight range: 0.136 - 6.223


## 10. Training Function with Early Stopping\n
\n
Training enhancements for transformers:\n
- **AdamW optimizer** - Adam with decoupled weight decay (better for transformers)\n
- **OneCycleLR scheduler** - Learning rate warmup + cosine annealing\n
- **Gradient clipping** - Prevents exploding gradients\n
- **Label smoothing** - Reduces overconfidence\n
- **Early stopping** - Stops when validation loss plateaus

In [20]:
from train_utils import create_training_components

components = create_training_components(
    model=model,
    train_labels=train_labels,  
    K=len(set(train_labels)),
    device=device,
    optimizer_type='adamw',
    lr=LEARNING_RATE,
    loss_type='label_smoothing_ce',
    scheduler_type='onecycle',
    weight_decay=WEIGHT_DECAY,
    label_smoothing=LABEL_SMOOTHING,
    max_lr=MAX_LR,
    epochs=NUM_EPOCHS,
    steps_per_epoch=len(train_loader)
)

# Loss function with label smoothing
    # criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=label_smoothing)
    
    # # AdamW optimizer (Adam with decoupled weight decay)
    # optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    # # OneCycleLR scheduler with warmup
    # scheduler = OneCycleLR(
    #     optimizer,
    #     max_lr=max_lr,
    #     epochs=num_epochs,
    #     steps_per_epoch=len(train_loader),
    #     pct_start=0.1,  # 10% warmup
    #     anneal_strategy='cos',
    #     div_factor=25.0,  # initial_lr = max_lr / 25
    #     final_div_factor=10000.0  # min_lr = initial_lr / 10000
    # )

In [21]:
def train_transformer(model, train_loader, val_loader, num_epochs, lr, max_lr, 
                     weight_decay, class_weights, label_smoothing, criterion, optimizer, scheduler,patience=7):
    """
    Train transformer model with advanced optimization techniques.
    
    Args:
        model: Transformer model
        train_loader: Training data loader
        val_loader: Validation data loader
        num_epochs: Maximum number of epochs
        lr: Initial learning rate
        max_lr: Maximum learning rate for OneCycleLR
        weight_decay: L2 regularization strength
        class_weights: Class weights for imbalanced data
        label_smoothing: Label smoothing factor
        patience: Early stopping patience
    
    Returns:
        Dictionary with model, history, and metrics
    """
    model.to(device)
    
    
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'lr': []}
    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    
    print("Training BERT-style Transformer...")
    start_time = time.time()
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        train_preds, train_labels_list = [], []
        
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
        for batch in train_pbar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            # Forward pass
            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            
            # Backward pass
            loss.backward()
            
            # Gradient clipping (important for transformers)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            scheduler.step()  # Update learning rate every batch
            
            # Track metrics
            train_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            train_preds.extend(preds.cpu().numpy())
            train_labels_list.extend(labels.cpu().numpy())
            
            # Update progress bar
            current_lr = scheduler.get_last_lr()[0]
            train_pbar.set_postfix({'loss': f'{loss.item():.4f}', 'lr': f'{current_lr:.2e}'})
        
        train_loss /= len(train_loader)
        train_acc = accuracy_score(train_labels_list, train_preds)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_preds, val_labels_list = [], []
        
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]"):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)
                
                outputs = model(input_ids, attention_mask)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                preds = torch.argmax(outputs, dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_labels_list.extend(labels.cpu().numpy())
        
        val_loss /= len(val_loader)
        val_acc = accuracy_score(val_labels_list, val_preds)
        
        # Record history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['lr'].append(scheduler.get_last_lr()[0])
        
        print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f} Train Acc={train_acc:.4f} | "
              f"Val Loss={val_loss:.4f} Val Acc={val_acc:.4f} | LR={scheduler.get_last_lr()[0]:.2e}")
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
            patience_counter = 0
            print(f"  -> New best model (val_loss: {val_loss:.4f})")
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break
    
    # Load best model
    model.load_state_dict(best_model_state)
    elapsed = time.time() - start_time
    print(f"Training completed in {elapsed/60:.2f} minutes")
    
    return {'model': model, 'history': history, 'best_val_loss': best_val_loss, 'time': elapsed}

## 11. Train Model

In [ ]:
# Train the transformer model
results = train_transformer(
    model,
    train_loader,
    val_loader,
    NUM_EPOCHS,
    LEARNING_RATE,
    MAX_LR,
    WEIGHT_DECAY,
    class_weights,
    LABEL_SMOOTHING,
    components['criterion'],
    components['optimizer'],
    components['scheduler'],
    EARLY_STOP_PATIENCE
)

## 12. Evaluate Model\n
\n
Evaluate the trained transformer on the test set with comprehensive metrics.

In [ ]:
def evaluate_transformer(model, loader):
    """Evaluate transformer model on a dataset."""
    model.eval()
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids, attention_mask)
            preds = torch.argmax(outputs, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Calculate metrics
    metrics = {
        'accuracy': accuracy_score(all_labels, all_preds),
        'precision': precision_score(all_labels, all_preds, average='macro', zero_division=0),
        'recall': recall_score(all_labels, all_preds, average='macro', zero_division=0),
        'f1_score': f1_score(all_labels, all_preds, average='macro', zero_division=0)
    }
    
    return metrics, all_preds, all_labels


# Evaluate on test set
test_metrics, test_preds, test_labels_list = evaluate_transformer(model, test_loader)

# Print results
print("=" * 80)
print("BERT-STYLE TRANSFORMER - TEST SET PERFORMANCE")
print("=" * 80)
print(f"Accuracy:  {test_metrics['accuracy']:.4f}")
print(f"Precision: {test_metrics['precision']:.4f}")
print(f"Recall:    {test_metrics['recall']:.4f}")
print(f"F1-Score:  {test_metrics['f1_score']:.4f}")
print("=" * 80)

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(results['history']['train_loss'], label='Train Loss', marker='o')
axes[0].plot(results['history']['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curves
axes[1].plot(results['history']['train_acc'], label='Train Accuracy', marker='o')
axes[1].plot(results['history']['val_acc'], label='Val Accuracy', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'transformer_bert_k{K}_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

# Plot learning rate schedule
plt.figure(figsize=(10, 4))
plt.plot(results['history']['lr'])
plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.title('Learning Rate Schedule (OneCycleLR with Warmup)')
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.tight_layout()
plt.savefig(f'transformer_bert_k{K}_lr_schedule.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Save Model\n
\n
Save the trained transformer model with full configuration and metrics.

In [ ]:
# Save checkpoint
checkpoint = {
    'model_type': 'BERT-style Transformer',
    'model_state_dict': model.state_dict(),
    'config': {
        'K': K,
        'vocab_size': len(vocab),
        'embedding_dim': EMBEDDING_DIM,
        'num_heads': NUM_HEADS,
        'num_layers': NUM_LAYERS,
        'hidden_dim': HIDDEN_DIM,
        'max_seq_len': MAX_SEQ_LEN,
        'dropout_rate': DROPOUT_RATE,
        'label_smoothing': LABEL_SMOOTHING,
        'weight_decay': WEIGHT_DECAY
    },
    'char_to_label': char_to_label,
    'label_to_char': label_to_char,
    'vocab': {
        'word2idx': vocab.word2idx,
        'idx2word': vocab.idx2word
    },
    'test_metrics': test_metrics,
    'history': results['history'],
    'training_time': results['time'],
    'total_parameters': sum(p.numel() for p in model.parameters())
}

filename = f'transformer_bert_k{K}.pt'
torch.save(checkpoint, filename)

print(f"Model saved to: {filename}")
print(f"\nModel Summary:")
print(f"  Architecture: BERT-style Transformer Encoder")
print(f"  Parameters: {checkpoint['total_parameters']:,}")
print(f"  Embedding dim: {EMBEDDING_DIM}")
print(f"  Attention heads: {NUM_HEADS}")
print(f"  Transformer layers: {NUM_LAYERS}")
print(f"  Feedforward dim: {HIDDEN_DIM}")
print(f"  Vocabulary size: {len(vocab):,}")
print(f"  Max sequence length: {MAX_SEQ_LEN}")
print(f"\nPerformance:")
print(f"  Test Accuracy: {test_metrics['accuracy']:.4f}")
print(f"  Test F1-Score: {test_metrics['f1_score']:.4f}")
print(f"  Training time: {results['time']/60:.2f} minutes")
print(f"\nComparison with other models:")
print(f"  Load this checkpoint and compare metrics with:")
print(f"    - baseline_logistic_regression.ipynb")
print(f"    - rnn_lstm_gru_models.ipynb")
print(f"    - seq2seq_basic.ipynb")